In [1]:
import numpy as np 
import pandas as pd 
import json

In [2]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
google_api_key = user_secrets.get_secret("Google_API_KEY")

from google import genai
client = genai.Client(api_key=google_api_key)

In [3]:
test_data_df = pd.read_csv("/kaggle/input/notebooks/zygong1994/finetuning-experiment-0326-etl/test.csv")

In [4]:
sys_prompt = '''
    You are an expert sales analyst. 
    Given the following sales conversation between a customer and a sales representative, 
    estimate the probability (0-1) that this customer will convert (i.e., purchase the product/sign up for service)
'''

# Batch Inference

In [5]:
from google.genai import types

text_strings = []
gemini_probs = []

for idx, row in test_data_df.iterrows():
    convo = json.loads(row['conversation'])
    text = "\n".join(f"{turn['speaker']}: {turn['message']}" for turn in convo)
    text_strings.append(text)
    
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=sys_prompt + text,
        config=types.GenerateContentConfig(
            temperature=0,
            response_mime_type="application/json",
            response_schema={
                "type": "object",
                "properties": {
                    "probability": {
                        "type": "number",
                        "description": "Conversion probability 0-1"
                    }
                },
                "required": ["probability"]
            }
        )
    )
    
    result = json.loads(response.text)
    gemini_probs.append(result['probability'])
    
    if (idx + 1) % 10 == 0:
        print(f"Processed {idx + 1}/{len(test_data_df)}")

print(f"Done. Processed {len(test_data_df)} conversations.")

Processed 10/200
Processed 20/200
Processed 30/200
Processed 40/200
Processed 50/200
Processed 60/200
Processed 70/200
Processed 80/200
Processed 90/200
Processed 100/200
Processed 110/200
Processed 120/200
Processed 130/200
Processed 140/200
Processed 150/200
Processed 160/200
Processed 170/200
Processed 180/200
Processed 190/200
Processed 200/200
Done. Processed 200 conversations.


In [6]:
results_df = pd.DataFrame({
    'text_string_conversation': text_strings,
    'ground_truth_conversion': test_data_df['outcome'].values,
    'gemini_probability': gemini_probs
})

results_df.head()
results_df.to_csv("Results_one_shot_Gemini.csv")

# Evaluation

In [7]:
from sklearn.metrics import roc_auc_score

y_true = results_df['ground_truth_conversion'].values
y_prob = results_df['gemini_probability'].values

# AUC-ROC
auc_roc = roc_auc_score(y_true, y_prob)
print(f"AUC-ROC: {auc_roc:.4f}")

# Cross-entropy: H(delta_y, p) = -[y * log(p) + (1-y) * log(1-p)]
# Clip to avoid log(0)
eps = 1e-15
y_prob_clipped = np.clip(y_prob, eps, 1 - eps)
cross_entropy = -np.mean(y_true * np.log(y_prob_clipped) + (1 - y_true) * np.log(1 - y_prob_clipped))
print(f"Binary Cross-Entropy: {cross_entropy:.4f}")

AUC-ROC: 0.8089
Binary Cross-Entropy: 0.5468
